# Screws

**Type:** Producer — stores output in dimensional depot  
**Blueprint:** `screw` (40 Iron Ingots/min → 160 Screws/min at 100%)  
**Internal chain:** Ingots → Iron Rods (3 constructors) → Screws (4 constructors)  
**Supply:** 309.90 of 810 total Iron Ingots/min (split across ingot lines)  
**Downstream:** Rotors and Reinforced Iron Plates both pull from this storage

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path('../..').resolve()))

from blueprints import BLUEPRINTS, Blueprint, Stage
from satisfactory import ITEMS

BP  = BLUEPRINTS
TOL = 0.5

## Constants

> Two downstream notebooks (Rotors and Reinforced Iron Plates) both consume from this storage.  
> Their combined pull must not exceed `screw_total − STASH_RATE` = `available_for_downstream`.

In [2]:
# ── Supply ────────────────────────────────────────────────────────────────────
IRON_INGOTS = 309.90  # items/min

# ── Stash target ──────────────────────────────────────────────────────────────
STASH_RATE = 50.0   # Screws/min accumulating in storage

# ── Blueprint placement ───────────────────────────────────────────────────────
SCREW_COPIES      = 8
SCREW_OUTPUT_EACH = 154.95  # Screws/min — total per blueprint copy

## Derived rates and balance

In [3]:
screw_total  = SCREW_COPIES * SCREW_OUTPUT_EACH
screw_ingots = screw_total  * BP['screw'].input_ratio('iron-ingot', 'screw')

available_for_downstream = screw_total - STASH_RATE

assert abs(screw_ingots - IRON_INGOTS) < TOL, (
    f"Ingot balance: consuming {screw_ingots:.2f}/min, supplied {IRON_INGOTS:.2f}/min "
    f"(delta {screw_ingots - IRON_INGOTS:+.2f})"
)
assert available_for_downstream > 0

# At 27.15 smart plating/min: rotors pull 32.16/min, RIPs pull 32.16/min
# Rotor blueprint: 25 screws per rotor → 32.16 × 25 = 804.00 screws/min
# RIP blueprint: 12 screws per RIP → 32.16 × 12 = 385.92 screws/min
rotors_share     = 804.00
reinforced_share = 385.92

print(f"✓ Chain balanced")
print(f"  {IRON_INGOTS:.2f} Iron Ingots/min  →  {screw_total:.2f} Screws/min")
print(f"  Stashing:                  {STASH_RATE:.0f} Screws/min")
print(f"  Available for downstream:  {available_for_downstream:.2f} Screws/min")
print(f"    → Rotors:                {rotors_share:.2f} Screws/min")
print(f"    → Reinforced Iron Plates:{reinforced_share:.2f} Screws/min")
print(f"    → Combined:              {rotors_share + reinforced_share:.2f} Screws/min")
print()
for r in BP['screw'].stage_rates('screw', SCREW_OUTPUT_EACH):
    name = ITEMS[r['item_key']].name
    print(f"  {r['machines']}× Constructor → {r['per_machine_rate']:.2f} {name}/min each")

✓ Chain balanced
  309.90 Iron Ingots/min  →  1239.60 Screws/min
  Stashing:                  50 Screws/min
  Available for downstream:  1189.60 Screws/min
    → Rotors:                804.00 Screws/min
    → Reinforced Iron Plates:385.92 Screws/min
    → Combined:              1189.92 Screws/min

  3× Constructor → 12.91 Iron Rod/min each
  4× Constructor → 38.74 Screw/min each
